In [ ]:
"""
Avellaneda-Stoikov Market Making Model

Paper : Avellaneda & Stoikov (2008) "High-frequency trading in a limit order book"
        Quantitative Finance, Vol.8, No.3

Closed-form solution from:
        Gueant, Lehalle & Fernandez-Tapia (2012) "Dealing with inventory risk"
        Mathematics and Financial Economics, Vol.7, No.4

PART 1  Faithful replication of the AS model with GLF closed-form quotes
PART 2  Three original extensions that address real desk weaknesses:
        (a) GARCH(1,1) online vol estimator  -- replaces constant sigma
        (b) OFI adverse selection filter     -- replaces uninformed-flow assumption
        (c) Hard inventory skew pressure     -- replaces unconstrained inventory

The base AS model is deployed daily on equity, crypto, and FX market-making desks.
This extension makes it usable in volatile, adversarial microstructure environments.
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")

#  GLOBAL PARAMETERS  (calibrated to 1-day HF session)

T        = 1.0          # session length (normalised)
DT       = 0.005        # time step size
N_STEPS  = int(T / DT)  # 200 steps per session
SIGMA    = 0.018        # base mid-price vol per step (~28% annual equivalent)
GAMMA    = 0.08         # risk aversion
K        = 1.5          # order-book depth decay
A        = 130.0        # baseline arrival intensity
Q_MAX    = 10           # hard inventory cap (lots)
S0       = 100.0        # initial mid-price

# Adverse scenario parameters (for Part 2 stress test)
TOXIC_LAMBDA = 6.0      # extra buy intensity from informed trader
VOL_SHOCK_T  = 0.38     # time of volatility regime shift
VOL_MULT     = 2.8      # sigma multiplier during shock



#  PART 1 -- AVELLANEDA-STOIKOV with GLF CLOSED-FORM QUOTES


class AvellanedaStoikov:
    """
    Optimal market-making under CARA (exponential) utility.

    The Hamilton-Jacobi-Bellman equation is solved by Gueant, Lehalle,
    Fernandez-Tapia (2012) in closed form, giving:

    Reservation price (indifference price):
        r(s, q, t) = s  -  q * gamma * sigma^2 * (T - t)

    Optimal bid-ask spread (GLF approximation):
        spread* = gamma * sigma^2 * (T - t)  +  (2/gamma) * ln(1 + gamma/k)

    The spread has two components:
        - Time-varying inventory risk term: shrinks to zero as T approaches
        - Liquidity (market depth) term: constant, depends only on k
    """

    def __init__(self, gamma=GAMMA, sigma=SIGMA, k=K, A=A,
                 T=T, dt=DT, q_max=Q_MAX):
        self.gamma = gamma
        self.sigma = sigma
        self.k     = k
        self.A     = A
        self.T     = T
        self.dt    = dt
        self.q_max = q_max

    def reservation_price(self, s, q, t):
        """Inventory-adjusted indifference price."""
        return s - q * self.gamma * self.sigma**2 * (self.T - t)

    def half_spread(self, t):
        """GLF closed-form optimal half-spread."""
        inv_risk = self.gamma * self.sigma**2 * (self.T - t) / 2.0
        liquidity = (1.0 / self.k) * np.log(1.0 + self.gamma / self.k)
        return inv_risk + liquidity

    def quotes(self, s, q, t):
        r  = self.reservation_price(s, q, t)
        hs = self.half_spread(t)
        return r - hs, r + hs       # bid, ask

    def fill_intensity(self, delta):
        """Poisson fill rate for a quote at distance delta from mid."""
        return self.A * np.exp(-self.k * delta)

    def simulate(self, n_paths=1, seed=42, inject_toxic=False):
        rng = np.random.default_rng(seed)
        results = []

        for p in range(n_paths):
            s = S0; q = 0.0; cash = 0.0

            s_path, q_path, pnl_path = [s], [q], [0.0]
            spread_path, res_path = [], []

            for i in range(N_STEPS):
                t    = i * self.dt
                q_c  = float(np.clip(q, -self.q_max, self.q_max))
                bid, ask = self.quotes(s, q_c, t)

                lam_b = self.fill_intensity(s - bid)
                lam_a = self.fill_intensity(ask - s)

                # extra buy pressure from informed trader if toxic scenario
                if inject_toxic and 0.2 < t < 0.7:
                    lam_b += TOXIC_LAMBDA

                bh = min(rng.poisson(lam_b * self.dt),
                         max(0, self.q_max - int(q_c)))
                sh = min(rng.poisson(lam_a * self.dt),
                         max(0, self.q_max + int(q_c)))

                q    += bh - sh
                cash += sh * ask - bh * bid

                # mid-price diffusion with vol shock
                shock = VOL_MULT if VOL_SHOCK_T < t < VOL_SHOCK_T + 0.06 else 1.0
                s    += rng.standard_normal() * self.sigma * shock

                s_path.append(s); q_path.append(q)
                pnl_path.append(cash + q * s)
                spread_path.append(ask - bid)
                res_path.append(self.reservation_price(s, q_c, t))

            results.append(dict(s=np.array(s_path), q=np.array(q_path),
                                pnl=np.array(pnl_path),
                                spread=np.array(spread_path),
                                res=np.array(res_path),
                                final_pnl=pnl_path[-1],
                                final_q=q_path[-1]))
        return results



#  PART 2 -- EXTENSIONS


class GARCH11:
    """Online GARCH(1,1) variance estimator (no fitting, just recursive update)."""
    def __init__(self, omega=2e-7, alpha=0.12, beta=0.86):
        self.omega = omega
        self.alpha = alpha
        self.beta  = beta
        self.h     = SIGMA**2

    def update(self, ret):
        self.h = self.omega + self.alpha * ret**2 + self.beta * self.h
        self.h = np.clip(self.h, 1e-10, (SIGMA * 5)**2)
        return np.sqrt(self.h)

    @property
    def sigma(self):
        return np.sqrt(self.h)


class AdaptiveMarketMaker(AvellanedaStoikov):
    """
    Three original extensions added on top of the AS/GLF framework:

    Extension 1 -- GARCH(1,1) volatility
        Each step the model updates a running variance estimate using the
        observed mid-price return. The reservation price and spread are
        recalculated using this live sigma, so the model widens quotes
        immediately when a vol regime shift is detected.

    Extension 2 -- OFI adverse-selection filter
        Order Flow Imbalance = (buy_fills - sell_fills) / total_fills
        When OFI exceeds a threshold (indicating correlated one-sided flow
        from an informed trader), the model asymmetrically widens the
        quote on the toxic side and tightens the other side. This reduces
        adverse selection without abandoning the market entirely.

    Extension 3 -- Inventory skew pressure
        Near the hard inventory limit the model applies an additional
        reservation price adjustment proportional to inventory/q_max.
        This creates a smooth penalty that discourages building large
        positions before hitting the hard clamp.
    """

    def __init__(self, ofi_window=15, tox_threshold=0.18,
                 tox_scale=0.5, skew_scale=0.4, **kwargs):
        super().__init__(**kwargs)
        self.ofi_window    = ofi_window
        self.tox_threshold = tox_threshold
        self.tox_scale     = tox_scale
        self.skew_scale    = skew_scale

    def _compute_ofi(self, buy_hist, sell_hist):
        b = sum(buy_hist[-self.ofi_window:])
        s = sum(sell_hist[-self.ofi_window:])
        return (b - s) / (b + s + 1e-9)

    def adaptive_quotes(self, s, q, t, garch, buy_hist, sell_hist):
        sig  = garch.sigma
        tau  = self.T - t
        ofi  = self._compute_ofi(buy_hist, sell_hist)

        # reservation price with live sigma + inventory skew
        inv_ratio = q / self.q_max
        r_base    = s - q * self.gamma * sig**2 * tau
        r         = r_base - self.skew_scale * self.half_spread(t) * inv_ratio

        # GLF half-spread with live sigma
        hs = self.gamma * sig**2 * tau / 2.0 + (1.0 / self.k) * np.log(1.0 + self.gamma / self.k)

        # OFI toxicity adjustment
        tox_mag = max(0, abs(ofi) - self.tox_threshold) * self.tox_scale
        if ofi > self.tox_threshold:      # buyers dominating: widen ask
            ask = r + hs + tox_mag
            bid = r - hs * 0.85
        elif ofi < -self.tox_threshold:   # sellers dominating: widen bid
            bid = r - hs - tox_mag
            ask = r + hs * 0.85
        else:
            bid = r - hs
            ask = r + hs

        return bid, ask, sig, ofi

    def simulate(self, n_paths=1, seed=42, inject_toxic=False):
        rng = np.random.default_rng(seed)
        results = []

        for p in range(n_paths):
            s = S0; q = 0.0; cash = 0.0
            g = GARCH11()
            buy_hist  = [0] * self.ofi_window
            sell_hist = [0] * self.ofi_window

            s_path, q_path, pnl_path = [s], [q], [0.0]
            spread_path, sig_path, ofi_path = [], [], []

            for i in range(N_STEPS):
                t   = i * self.dt
                q_c = float(np.clip(q, -self.q_max, self.q_max))

                bid, ask, sig_now, ofi = self.adaptive_quotes(
                    s, q_c, t, g, buy_hist, sell_hist)

                lam_b = self.fill_intensity(s - bid)
                lam_a = self.fill_intensity(ask - s)

                if inject_toxic and 0.2 < t < 0.7:
                    lam_b += TOXIC_LAMBDA

                bh = min(rng.poisson(lam_b * self.dt),
                         max(0, self.q_max - int(q_c)))
                sh = min(rng.poisson(lam_a * self.dt),
                         max(0, self.q_max + int(q_c)))

                buy_hist.append(bh); sell_hist.append(sh)
                q    += bh - sh
                cash += sh * ask - bh * bid

                shock = VOL_MULT if VOL_SHOCK_T < t < VOL_SHOCK_T + 0.06 else 1.0
                ret   = rng.standard_normal() * self.sigma * shock
                s    += ret
                g.update(ret / S0)

                s_path.append(s); q_path.append(q)
                pnl_path.append(cash + q * s)
                spread_path.append(ask - bid)
                sig_path.append(sig_now)
                ofi_path.append(ofi)

            results.append(dict(s=np.array(s_path), q=np.array(q_path),
                                pnl=np.array(pnl_path),
                                spread=np.array(spread_path),
                                sigma=np.array(sig_path),
                                ofi=np.array(ofi_path),
                                final_pnl=pnl_path[-1],
                                final_q=q_path[-1]))
        return results



#  VISUALISATION


C = dict(blue="#185FA5", teal="#0F6E56", coral="#D85A30", amber="#BA7517",
         gray="#5F5E5A", lgray="#D3D1C7", purple="#534AB7", red="#A32D2D")

def plot_all():
    base = AvellanedaStoikov()
    ext  = AdaptiveMarketMaker()

    # Normal scenario
    b1 = base.simulate(n_paths=1, seed=7)[0]
    e1 = ext.simulate( n_paths=1, seed=7)[0]

    # Toxic + vol shock scenario
    b2 = base.simulate(n_paths=1, seed=7, inject_toxic=True)[0]
    e2 = ext.simulate( n_paths=1, seed=7, inject_toxic=True)[0]

    # Monte Carlo (adverse scenario)
    b_mc = base.simulate(n_paths=400, seed=99, inject_toxic=True)
    e_mc = ext.simulate( n_paths=400, seed=99, inject_toxic=True)
    b_pnl = np.array([r["final_pnl"] for r in b_mc])
    e_pnl = np.array([r["final_pnl"] for r in e_mc])
    b_q   = np.array([r["final_q"]   for r in b_mc])
    e_q   = np.array([r["final_q"]   for r in e_mc])

    t_step = np.linspace(0, T, N_STEPS + 1)
    t_spr  = np.linspace(0, T, N_STEPS)

    fig = plt.figure(figsize=(20, 15), facecolor="white")
    fig.suptitle(
        "Avellaneda-Stoikov Market Making  |  Replication + GARCH-Vol / OFI Adverse Selection Extension",
        fontsize=14, fontweight="bold", y=0.985, color="#2C2C2A"
    )
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.46, wspace=0.36)

    def sty(ax, title, xl, yl):
        ax.set_title(title, fontsize=11, fontweight="bold", pad=5)
        ax.set_xlabel(xl, fontsize=9); ax.set_ylabel(yl, fontsize=9)
        ax.grid(True, alpha=0.22, lw=0.7); ax.tick_params(labelsize=8)

    # 1. Quotes and reservation price (normal)
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(t_step, b1["s"],   color=C["gray"],  lw=1.0, label="mid-price", zorder=2)
    ax1.plot(t_step[:-1], b1["res"], color=C["blue"], lw=1.5, ls="--", label="reservation px", zorder=3)
    bid_b = b1["res"] - b1["spread"] / 2
    ask_b = b1["res"] + b1["spread"] / 2
    ax1.fill_between(t_step[:-1], bid_b, ask_b, alpha=0.22, color=C["teal"], label="bid-ask band")
    sty(ax1, "Reservation price and quotes (AS base)", "time", "price ($)")
    ax1.legend(fontsize=7.5, framealpha=0)

    # 2. Inventory under toxic flow: base vs extension
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(t_step, b2["q"], color=C["blue"],  lw=2.0, label="AS base (toxic)")
    ax2.plot(t_step, e2["q"], color=C["teal"],  lw=2.0, label="adaptive (toxic)")
    ax2.axhline(0, color=C["gray"], lw=0.7, ls="--")
    ax2.axhspan( Q_MAX-0.5,  Q_MAX+0.5, alpha=0.12, color=C["red"])
    ax2.axhspan(-Q_MAX-0.5, -Q_MAX+0.5, alpha=0.12, color=C["red"])
    ax2.axvline(0.2, color=C["amber"], lw=1.0, ls=":", alpha=0.8)
    ax2.axvline(0.7, color=C["amber"], lw=1.0, ls=":", alpha=0.8)
    ax2.text(0.45, 9.2, "toxic flow window", fontsize=7, color=C["amber"], ha="center")
    sty(ax2, "Inventory under informed order flow", "time", "inventory (lots)")
    ax2.legend(fontsize=7.5, framealpha=0)

    # 3. GARCH sigma vs constant sigma
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.axhline(SIGMA, color=C["blue"], lw=1.5, ls="--", label="constant sigma (AS)")
    ax3.plot(t_spr, e1["sigma"], color=C["teal"], lw=2.0, label="GARCH(1,1) sigma")
    ax3.axvspan(VOL_SHOCK_T, VOL_SHOCK_T + 0.06, alpha=0.15, color=C["coral"], label="vol shock")
    ax3.fill_between(t_spr, e1["sigma"], SIGMA,
                     where=(e1["sigma"] > SIGMA), alpha=0.2, color=C["coral"])
    sty(ax3, "Volatility: constant vs GARCH (ext.)", "time", "sigma per step")
    ax3.legend(fontsize=7.5, framealpha=0)

    # 4. Spread comparison (normal vs toxic, base vs adaptive)
    ax4 = fig.add_subplot(gs[1, 0])
    ax4.plot(t_spr, b1["spread"], color=C["blue"],   lw=1.3, alpha=0.8, label="base (normal)")
    ax4.plot(t_spr, e1["spread"], color=C["teal"],   lw=1.3, alpha=0.8, label="adaptive (normal)")
    ax4.plot(t_spr, b2["spread"], color=C["blue"],   lw=1.8, ls="--",   label="base (toxic)")
    ax4.plot(t_spr, e2["spread"], color=C["coral"],  lw=2.0,            label="adaptive (toxic)")
    ax4.axvspan(0.2, 0.7, alpha=0.06, color=C["amber"])
    ax4.axvspan(VOL_SHOCK_T, VOL_SHOCK_T + 0.06, alpha=0.12, color=C["red"])
    sty(ax4, "Bid-ask spread: base vs adaptive", "time", "spread ($)")
    ax4.legend(fontsize=7.5, framealpha=0, ncol=2)

    # 5. OFI signal (extension)
    ax5 = fig.add_subplot(gs[1, 1])
    ofi = e2["ofi"]
    cols = [C["coral"] if v > 0.18 else (C["blue"] if v < -0.18 else C["lgray"]) for v in ofi]
    ax5.bar(t_spr, ofi, width=DT * 0.85, color=cols, alpha=0.8)
    ax5.axhline( 0.18, color=C["red"], ls="--", lw=0.9, label="toxicity threshold")
    ax5.axhline(-0.18, color=C["red"], ls="--", lw=0.9)
    ax5.axhline(0, color=C["gray"], lw=0.5)
    ax5.axvspan(0.2, 0.7, alpha=0.06, color=C["amber"])
    sty(ax5, "OFI adverse-selection signal (ext.)", "time", "OFI [-1, +1]")
    ax5.legend(fontsize=7.5, framealpha=0)

    # 6. PnL comparison under adverse scenario
    ax6 = fig.add_subplot(gs[1, 2])
    ax6.plot(t_step, b2["pnl"], color=C["blue"],  lw=2.0, label="AS base (toxic)")
    ax6.plot(t_step, e2["pnl"], color=C["teal"],  lw=2.0, label="adaptive (toxic)")
    ax6.axhline(0, color=C["gray"], lw=0.7, ls="--")
    ax6.axvspan(0.2, 0.7, alpha=0.06, color=C["amber"])
    sty(ax6, "P&L under informed order flow", "time", "mark-to-market P&L ($)")
    ax6.legend(fontsize=7.5, framealpha=0)

    # 7. PnL distribution (MC adverse scenario)
    ax7 = fig.add_subplot(gs[2, 0])
    lo = min(b_pnl.min(), e_pnl.min()) - 0.5
    hi = max(b_pnl.max(), e_pnl.max()) + 0.5
    bins = np.linspace(lo, hi, 45)
    ax7.hist(b_pnl, bins=bins, color=C["blue"],  alpha=0.6,
             label=f"AS  mean={b_pnl.mean():.2f}  std={b_pnl.std():.2f}")
    ax7.hist(e_pnl, bins=bins, color=C["teal"],  alpha=0.6,
             label=f"Ext mean={e_pnl.mean():.2f}  std={e_pnl.std():.2f}")
    sty(ax7, "P&L distribution (MC, 400 paths, toxic)", "final P&L ($)", "count")
    ax7.legend(fontsize=7.5, framealpha=0)

    # 8. Final inventory distribution (MC adverse scenario)
    ax8 = fig.add_subplot(gs[2, 1])
    lim = int(max(abs(b_q).max(), abs(e_q).max())) + 2
    bins_q = np.arange(-lim - 0.5, lim + 1.5, 1)
    ax8.hist(b_q, bins=bins_q, color=C["blue"],  alpha=0.6, rwidth=0.55,
             label=f"AS   std={b_q.std():.2f}")
    ax8.hist(e_q, bins=bins_q, color=C["teal"],  alpha=0.6, rwidth=0.55,
             label=f"Ext  std={e_q.std():.2f}")
    ax8.axvline(0, color=C["gray"], lw=1.0, ls="--")
    sty(ax8, "End-of-day inventory (MC, toxic)", "inventory (lots)", "count")
    ax8.legend(fontsize=7.5, framealpha=0)

    # 9. Reservation price adjustment as function of q and time
    ax9 = fig.add_subplot(gs[2, 2])
    q_vals = np.linspace(-Q_MAX, Q_MAX, 200)
    for tv, col, lbl in [(0.1, C["coral"],"t=0.1"), (0.5, C["blue"],"t=0.5"), (0.9, C["teal"],"t=0.9")]:
        adj = -q_vals * GAMMA * SIGMA**2 * (T - tv)
        ax9.plot(q_vals, adj, color=col, lw=1.8, label=lbl)
    ax9.axhline(0, color=C["gray"], lw=0.5, ls="--")
    ax9.axvline(0, color=C["gray"], lw=0.5, ls="--")
    sty(ax9, "Reservation price skew (q vs time)", "inventory (lots)", "price adj. ($)")
    ax9.legend(fontsize=7.5, framealpha=0)

    

    out = "avellaneda_stoikov_extended.png"
    plt.savefig(out, dpi=155, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"Saved: {out}")



#  SUMMARY


def print_summary():
    base = AvellanedaStoikov()
    ext  = AdaptiveMarketMaker()

    # Compare under ADVERSE (toxic + vol shock) scenario
    b_mc = base.simulate(n_paths=600, seed=42, inject_toxic=True)
    e_mc = ext.simulate( n_paths=600, seed=42, inject_toxic=True)

    bp = np.array([r["final_pnl"] for r in b_mc])
    ep = np.array([r["final_pnl"] for r in e_mc])
    bq = np.array([r["final_q"]   for r in b_mc])
    eq = np.array([r["final_q"]   for r in e_mc])

    def bps_sharpe(pnl): return pnl.mean() / (pnl.std() + 1e-9)

  
    print("  AVELLANEDA-STOIKOV BASE MODEL (Part 1)")
    print("  Scenario: toxic order flow + vol regime shift")
   
    print(f"  Mean final P&L    : ${bp.mean():.3f}")
    print(f"  P&L std dev       : ${bp.std():.3f}")
    print(f"  Sharpe (session)  : {bps_sharpe(bp):.3f}")
    print(f"  End inv mean/std  : {bq.mean():.2f} / {bq.std():.2f} lots")
    print(f"  P&L 5th pctile    : ${np.percentile(bp, 5):.3f}")
    
    print("  ADAPTIVE EXTENSION (Part 2)")
    
    print(f"  Mean final P&L    : ${ep.mean():.3f}")
    print(f"  P&L std dev       : ${ep.std():.3f}")
    print(f"  Sharpe (session)  : {bps_sharpe(ep):.3f}")
    print(f"  End inv mean/std  : {eq.mean():.2f} / {eq.std():.2f} lots")
    print(f"  P&L 5th pctile    : ${np.percentile(ep, 5):.3f}")
    inv_red    = (bq.std() - eq.std()) / bq.std() * 100
    tail_imp   = (np.percentile(ep, 5) - np.percentile(bp, 5))
    sharpe_imp = (bps_sharpe(ep) - bps_sharpe(bp)) / abs(bps_sharpe(bp)) * 100
    
    print(f"  Inventory risk reduction  : {inv_red:.1f}%")
    print(f"  5th-pctile P&L lift       : +${tail_imp:.3f}")
    print(f"  Sharpe change             : {sharpe_imp:+.1f}%")
    print("=" * 62)


if __name__ == "__main__":
    print_summary()
    
    plot_all()
    

  AVELLANEDA-STOIKOV BASE MODEL (Part 1)
  Scenario: toxic order flow + vol regime shift
  Mean final P&L    : $8.258
  P&L std dev       : $1.766
  Sharpe (session)  : 4.675
  End inv mean/std  : 0.48 / 6.05 lots
  P&L 5th pctile    : $5.259
  ADAPTIVE EXTENSION (Part 2)
--------------------------------------------------------------
  Mean final P&L    : $11.442
  P&L std dev       : $2.066
  Sharpe (session)  : 5.538
  End inv mean/std  : 0.20 / 6.05 lots
  P&L 5th pctile    : $7.924
  Inventory risk reduction  : -0.1%
  5th-pctile P&L lift       : +$2.665
  Sharpe change             : +18.4%
Generating figure ...
Saved: avellaneda_stoikov_extended.png
Done.
